# **8.1 Reconocimiento de Dígitos Manuscritos con Keras (CNN)**

Este cuaderno implementa una **Red Neuronal Convolucional (CNN)** para reconocer dígitos manuscritos del 0 al 9 usando el dataset **MNIST**, siguiendo el mismo esquema que el ejemplo de clasificación de prendas de ropa (Fashion-MNIST).

**Pasos del ejercicio:**
1. Instalación de dependencias
2. Importar librerías
3. Cargar los datos
4. Preprocesamiento
5. Creación de la red neuronal (CNN)
6. Entrenamiento
7. Evaluación
8. Uso: predicciones sobre imágenes nuevas

---
## **Paso 1 – Instalación de Dependencias**

Las librerías necesarias (TensorFlow/Keras, NumPy y Matplotlib) suelen estar ya disponibles en Google Colab. Si ejecutas este cuaderno en un entorno local, instálalas con el siguiente comando.

In [ ]:
# Descomenta la línea siguiente si ejecutas en local (en Colab no es necesario)
# !pip install tensorflow numpy matplotlib

---
## **Paso 2 – Importar Librerías**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow versión: {tf.__version__}")

---
## **Paso 3 – Cargar los Datos**

Usamos el dataset **MNIST** incluido en Keras. Contiene **70 000 imágenes en escala de grises** de dígitos manuscritos (28×28 píxeles):
- **60 000** imágenes para entrenamiento
- **10 000** imágenes para prueba

Las etiquetas son los dígitos del **0 al 9** (10 clases en total).

In [ ]:
# Cargar el dataset MNIST
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print(f"Forma de X_train : {X_train.shape}  → {X_train.shape[0]} imágenes de {X_train.shape[1]}×{X_train.shape[2]} px")
print(f"Forma de X_test  : {X_test.shape}")
print(f"Forma de y_train : {y_train.shape}")
print(f"Clases disponibles: {np.unique(y_train)}")

In [ ]:
# Visualizar algunos ejemplos del conjunto de entrenamiento
nombres_clases = [str(i) for i in range(10)]

plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[i], cmap='gray')
    plt.title(f"Dígito: {y_train[i]}")
    plt.axis('off')
plt.suptitle('Ejemplos del conjunto de entrenamiento (MNIST)', fontsize=13)
plt.tight_layout()
plt.show()

---
## **Paso 4 – Preprocesamiento**

Antes de alimentar la red neuronal, realizamos tres operaciones:

1. **Reshape**: añadimos la dimensión del canal de color (escala de grises = 1 canal) que exige la capa Conv2D → `(28, 28)` → `(28, 28, 1)`.
2. **Normalización**: escalamos los valores de píxel de `[0, 255]` a `[0.0, 1.0]` dividiendo entre 255.
3. **One-Hot Encoding**: convertimos las etiquetas numéricas a vectores binarios de 10 posiciones para usar `categorical_crossentropy`.

In [ ]:
# 1. Reshape: añadir canal de color
X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

# 2. Normalización: valores de píxel a [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32')  / 255.0

# 3. One-Hot Encoding de las etiquetas
num_clases = 10
y_train_ohe = keras.utils.to_categorical(y_train, num_clases)
y_test_ohe  = keras.utils.to_categorical(y_test,  num_clases)

print(f"X_train preprocesado : {X_train.shape}")
print(f"y_train original     : {y_train[0]}")
print(f"y_train One-Hot      : {y_train_ohe[0]}")

---
## **Paso 5 – Creación de la Red Neuronal (CNN)**

Construimos una **Red Neuronal Convolucional (CNN)** con el modelo `Sequential` de Keras, inspirada en la arquitectura del ejemplo de prendas de ropa:

| Capa | Tipo | Descripción |
|------|------|-------------|
| 1 | `Conv2D` (32 filtros, 3×3) + ReLU | Extrae características locales de las imágenes |
| 2 | `MaxPooling2D` (2×2) | Reduce dimensiones, mantiene características relevantes |
| 3 | `Conv2D` (64 filtros, 3×3) + ReLU | Extrae características más complejas |
| 4 | `MaxPooling2D` (2×2) | Segunda reducción de dimensiones |
| 5 | `Flatten` | Convierte el mapa de características en un vector 1D |
| 6 | `Dense` (128 neuronas) + ReLU | Capa densa de clasificación |
| 7 | `Dropout` (0.5) | Regularización para evitar sobreajuste |
| 8 | `Dense` (10 neuronas) + Softmax | Salida: probabilidad para cada dígito (0–9) |

**Función de coste**: `categorical_crossentropy` (adecuada para clasificación multiclase con One-Hot)  
**Optimizador**: `Adam` (adaptativo, eficiente y rápido en converger)

In [ ]:
# Crear el modelo CNN Sequential
model = keras.Sequential([

    # --- Bloque Convolucional 1 ---
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # --- Bloque Convolucional 2 ---
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # --- Aplanado + Capas Densas ---
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),          # Evitar sobreajuste
    layers.Dense(num_clases, activation='softmax')  # 10 clases
])

# Compilar el modelo
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Resumen de la arquitectura
model.summary()

---
## **Paso 6 – Entrenamiento**

Entrenamos la red con el método `fit`:
- **epochs = 10**: número de pasadas completas sobre el conjunto de entrenamiento.
- **batch_size = 64**: número de imágenes procesadas en cada iteración del optimizador.
- **validation_split = 0.1**: reservamos el 10 % del entrenamiento para monitorizar la generalización del modelo durante el entrenamiento.

In [ ]:
historial = model.fit(
    X_train, y_train_ohe,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Visualizar las curvas de entrenamiento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Pérdida (Loss)
ax1.plot(historial.history['loss'],     label='Entrenamiento')
ax1.plot(historial.history['val_loss'], label='Validación')
ax1.set_title('Función de Pérdida (Loss)')
ax1.set_xlabel('Época')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# Precisión (Accuracy)
ax2.plot(historial.history['accuracy'],     label='Entrenamiento')
ax2.plot(historial.history['val_accuracy'], label='Validación')
ax2.set_title('Precisión (Accuracy)')
ax2.set_xlabel('Época')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True)

plt.suptitle('Curvas de Entrenamiento – CNN MNIST', fontsize=13)
plt.tight_layout()
plt.show()

---
## **Paso 7 – Evaluación**

Evaluamos el modelo con las **10 000 imágenes de prueba** que nunca ha visto durante el entrenamiento. Obtenemos la **pérdida (loss)** y la **precisión (accuracy)**.

In [ ]:
loss_test, accuracy_test = model.evaluate(X_test, y_test_ohe, verbose=0)

print(f"\n{'='*45}")
print(f"  Resultados sobre el conjunto de PRUEBA")
print(f"{'='*45}")
print(f"  Loss (pérdida)    : {loss_test:.4f}")
print(f"  Accuracy (exactitud): {accuracy_test:.4f}  ({accuracy_test*100:.2f} %)")
print(f"{'='*45}")

In [ ]:
# Matriz de confusión
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred_prob = model.predict(X_test)
y_pred      = np.argmax(y_pred_prob, axis=1)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=nombres_clases)

fig, ax = plt.subplots(figsize=(8, 7))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matriz de Confusión – CNN MNIST', fontsize=13)
plt.tight_layout()
plt.show()

---
## **Paso 8 – Uso: Predicciones sobre Imágenes Nuevas**

Utilizamos el modelo entrenado para **predecir dígitos en imágenes que no ha visto antes**.

Para cada imagen se obtiene un **vector de 10 probabilidades** (una por dígito). La clase predicha es la de mayor probabilidad (`argmax`).

In [ ]:
# Seleccionar 10 imágenes aleatorias del conjunto de prueba
np.random.seed(42)
indices = np.random.choice(len(X_test), 10, replace=False)

imagenes  = X_test[indices]
etiquetas = y_test[indices]

# Obtener predicciones
predicciones_prob = model.predict(imagenes)
predicciones      = np.argmax(predicciones_prob, axis=1)

# Mostrar resultados
plt.figure(figsize=(14, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(imagenes[i].reshape(28, 28), cmap='gray')

    correcto = predicciones[i] == etiquetas[i]
    color    = 'green' if correcto else 'red'
    simbolo  = '✓' if correcto else '✗'

    plt.title(f"Real: {etiquetas[i]} | Pred: {predicciones[i]} {simbolo}",
              color=color, fontsize=9)
    plt.axis('off')

plt.suptitle('Predicciones sobre el conjunto de prueba\n(verde = correcto | rojo = error)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Mostrar la distribución de probabilidades para una imagen concreta
idx = 0  # cambia este índice para ver otras imágenes

plt.figure(figsize=(10, 3))

plt.subplot(1, 2, 1)
plt.imshow(imagenes[idx].reshape(28, 28), cmap='gray')
plt.title(f"Imagen – Dígito real: {etiquetas[idx]}")
plt.axis('off')

plt.subplot(1, 2, 2)
barras = plt.bar(range(10), predicciones_prob[idx], color='steelblue')
barras[predicciones[idx]].set_color('tomato')
plt.xticks(range(10))
plt.xlabel('Dígito')
plt.ylabel('Probabilidad')
plt.title(f"Distribución de probabilidades\nPredicción: {predicciones[idx]} ({predicciones_prob[idx][predicciones[idx]]*100:.1f} %)")
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.show()

---
## **Reflexión Final**

| Parámetro | Valor utilizado | Motivo |
|-----------|----------------|--------|
| **Función de coste** | `categorical_crossentropy` | Clasificación multiclase con One-Hot Encoding |
| **Optimizador** | `Adam (lr=0.001)` | Adaptativo, rápido en converger, buena generalización |
| **Activación capas ocultas** | `ReLU` | Evita el problema del gradiente desvaneciente |
| **Activación capa de salida** | `Softmax` | Genera distribución de probabilidades sobre 10 clases |
| **Regularización** | `Dropout (0.5)` | Reduce el sobreajuste desactivando neuronas aleatoriamente |
| **Epochs** | `10` | Suficiente para convergir sin sobreajustar en MNIST |
| **Batch size** | `64` | Equilibrio entre velocidad y estabilidad del gradiente |

### ¿Cómo mejorar la precisión?
- Aumentar el número de épocas (p.ej. 20).
- Añadir más capas convolucionales o más filtros.
- Aplicar **Data Augmentation** (rotaciones, desplazamientos) con `ImageDataGenerator`.
- Usar **Batch Normalization** entre capas para estabilizar el entrenamiento.
- Reducir el `learning_rate` con un `ReduceLROnPlateau` callback.